# **Importing Necessary Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# **Load and Explore Data🔎**

In [ ]:
# IPL_Squad_2023 Dataset 
data = pd.read_csv('/kaggle/input/ipl-pre-auction-and-post-auction-2024/IPL 2024 Auction/IPL_Squad_2023.csv')
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.nunique()

In [ ]:
# drop unnamed: 0 column
data.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
data['Base Price']= pd.to_numeric(data['Base Price'], errors='coerce')

In [ ]:
#Categorize players
data['status']=data['COST IN ₹ (CR.)'].apply(lambda x: 'Retained' if x == 0 and pd.notna(x)
                else 'Bought' if x > 0 else 'Unsold')

In [ ]:
data.head()

In [ ]:
print("Column names in your dataset:")
print(data.columns.tolist())

# **Analysis📝 & Visualiation📊**

In [ ]:
# Total players per team(2023)
players_per_team = data.groupby('Team')['Player\'s List'].count().sort_values(ascending=False)
players_per_team

In [ ]:
players_per_team_2022 = data.groupby('2022 Squad')['Player\'s List'].count().sort_values(ascending=False)
players_per_team_2022



*   Team Delhi Capitals(DC) ---> bough 2 Players
*   Team Chennai Super Kings(CSK) ---> bough 1 Player
*   Team Gujarat Titans(GT) ---> bough 3 Players
*   Team Lucknow Super Giants(LSG)---> bough 4 Players
*   Team Rajasthan Royals(RR) ---> bough 1 Player
*   Team Royal Challengers Banglore(RCB) ---> bough 5 Players
*   Team Sunrisers Hyderabad(SRH) ---> bough 3 Players


*   Team Mumbai Indians(MI) ---> sold 4 Players
*   Team Punjab Super Kings(PBSK) ---> sold 2 Players



*   Team Kolkata Knight Riders(KKR) ---> Keep his squad











In [ ]:
# Team Spending Analysis
team_spend = data[data['status'] == 'Bought'].groupby('Team')['COST IN ₹ (CR.)'].sum().sort_values(ascending=False)
team_spend

In [ ]:
plt.figure(figsize=(12,6))
team_spend.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('IPL 2023 Auction: Total Team Expenditure', fontsize=16, fontweight='bold')
plt.xlabel('Team', fontsize=12)
plt.ylabel('Total Spend (₹ Crores)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Player Type Distribution
type_dist = data[data['status'] == 'Bought']['TYPE'].value_counts()
print(type_dist)

plt.figure(figsize=(10,6))
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99']
plt.pie(type_dist.values, labels=type_dist.index, autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('IPL 2023 Auction: Player Type Distribution', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.show()


In [ ]:
# Top 10 Most Expensive Players
top_10 = data[data['status'] == 'Bought'].nlargest(10, 'COST IN ₹ (CR.)')[['Player\'s List', 'Team', 'TYPE', 'COST IN ₹ (CR.)']]

print("\n TOP 10 MOST EXPENSIVE PLAYERS - IPL 2023 AUCTION")
print("="*60)
print(top_10.to_string(index=False))

In [ ]:
# UNSOLD PLAYERS ANALYSIS
unsold = data[data['status'] == 'Unsold']
print("\n UNSOLD PLAYERS ANALYSIS")
print("="*60)
print(f"Total Unsold Players: {len(unsold)}")
print(f"Most common unsold type: {unsold['TYPE'].mode()[0]}")
print(f"Average base price of unsold: ₹{unsold['Base Price'].mean()/1e6:.1f} Lakhs")


In [ ]:
# Retention rates by team
print("\n Retention Rate by Team:")
retention_rate = data.groupby('Team').apply(
    lambda x: (x['status'] == 'Retained').sum() / len(x) * 100
).sort_values(ascending=False).round(1)
print(retention_rate)


In [ ]:
# TEAM STRATEGY ANALYSIS
team_stats = data[data['status'] == 'Bought'].groupby('Team').agg({
    'Player\'s List': 'count',
    'COST IN ₹ (CR.)': 'sum',
    'TYPE': lambda x: x.value_counts().index[0] if len(x) > 0 else 'None'
}).rename(columns={'Player\'s List': 'Players Bought', 'COST IN ₹ (CR.)': 'Total Spend', 'TYPE': 'Most Common Type'})

team_retained = data[data['status'] == 'Retained'].groupby('Team').size().rename('Players Retained')

team_strategy = pd.concat([team_stats, team_retained], axis=1).fillna(0)
team_strategy['Avg Price'] = team_strategy['Total Spend'] / team_strategy['Players Bought']

print("\n📊 TEAM STRATEGY ANALYSIS")
print("="*60)
print(team_strategy.round(2))